In [2]:
!pip install transformers[torch] accelerate -U
!pip install emoji datasets scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
import emoji
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


file_path = 'final2055.csv'
df = pd.read_csv(file_path)


df = df[['comment', 'sentiment']].copy()
import re

def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'@\S+', '', text) # Remove mentions
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df['comment'] = df['comment'].apply(clean_text)

df['comment'] = df['comment'].astype(str)


le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Total Samples: {len(df)}")
print(f"Classes: {le.classes_}")
print(f"Training: {len(train_texts)}, Testing: {len(test_texts)}")

Total Samples: 2055
Classes: ['N' 'NEU' 'P']
Training: 1644, Testing: 411


In [ ]:

from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Re-create the dataset objects
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer)
print(f"Tokenization setup updated successfully for {MODEL_NAME}!")

Tokenization setup updated successfully for xlm-roberta-base!


In [5]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1
    }

In [ ]:

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch
from torch import nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "xlm-roberta-base" 

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# 3. Create Custom Trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# 4. Define Training Arguments (Set for 10 epochs)
training_args = TrainingArguments(
    output_dir='./results_xlm',          
    num_train_epochs=5,              
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,                  
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",               
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    warmup_steps=100,
    report_to="none",
    save_total_limit=1                  
)

# 5. Initialize the CustomTrainer
trainer = CustomTrainer(
    model=model,                         
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1343.09it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Asus\AppData\Roaming\Python\Pyt

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.066639,0.527981,0.447803


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.88s/it]
C:\Users\Asus\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)


final_acc = predictions.metrics['test_accuracy'] * 100
print(f"Final Optimized Accuracy: {final_acc:.2f}%")


print("\nFinal Classification Report:\n")
print(classification_report(test_labels, y_pred, target_names=le.classes_))


cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Sentiment')
plt.ylabel('Actual Sentiment')
plt.title('Final Confusion Matrix - XLM-RoBERTa (Optimized)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data for the models
models = ['SVM', 'Random Forest', 'Logistic Regression', 'XLM-RoBERTa']
accuracies = [68.13, 65.45, 68.86, 66.18]

# Colors for the bars
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'] # Blue, Orange, Green, Red

# Create the bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=colors, width=0.5)

# Set y-axis limit to match the original paper's graph
plt.ylim(60, 100)

# Add labels and title
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Performance Comparison of Machine Learning and Deep Learning Models', fontsize=14)

# Add accuracy text on top of each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval}%',
             ha='center', va='bottom', fontweight='bold')

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xlm)

plt.figure(figsize=(8, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['N', 'NEU', 'P'],
            yticklabels=['N', 'NEU', 'P'])

plt.xlabel('Predicted Sentiment', fontsize=12)
plt.ylabel('Actual Sentiment', fontsize=12)
plt.title('Confusion Matrix for XLM-RoBERTa', fontsize=14)
plt.show()